### Import necessary libraries and install required packages

In [9]:
!pip install transformers datasets scikit-learn torch pandas pyarrow

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm

# --------- data & tokenizer ----------
train_df = pd.read_csv("train.csv", sep=",")
test_df  = pd.read_csv("test.csv", sep=",")

label_names = ["Joy","Trust","Fear","Surprise","Sadness","Disgust","Anger","Anticipation","Neutral"]
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --------- custom Dataset (returns raw lists so collator can pad) ----------
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

train_dataset = EmotionDataset(train_df, tokenizer, label_names)
eval_dataset  = EmotionDataset(test_df, tokenizer, label_names)

# --------- collate function that uses HuggingFace padding and stacks labels ----------
def collate_fn(batch):
    # batch: list of dicts from __getitem__
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)                        # returns tensors for input_ids/attention_mask etc
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)    # shape: (batch, num_labels)
    return batch_inputs

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
eval_loader  = DataLoader(eval_dataset,  batch_size=16, shuffle=False, collate_fn=collate_fn)

# --------- model, optimizer, scheduler ----------
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# --------- training loop ----------
progress_bar = tqdm(range(num_training_steps))
model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

# --------- evaluation ----------
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in eval_loader:
        labels = batch["labels"].cpu().numpy().astype(int)   # (batch, num_labels)
        # move only the inputs to device
        model_inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)   # threshold 0.5
        all_preds.append(preds)
        all_labels.append(labels)

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

# metrics
print("Subset Accuracy (exact match):", accuracy_score(y_true, y_pred))
print("Micro Precision:", precision_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro Recall:",    recall_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro F1:",        f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro Precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Macro Recall:",    recall_score(y_true, y_pred, average="macro", zero_division=0))
print("Macro F1:",        f1_score(y_true, y_pred, average="macro", zero_division=0))
print("\nPer-class report:\n", classification_report(y_true, y_pred, target_names=label_names, zero_division=0))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 327/327 [05:20<00:00,  1.02it/s]

100%|██████████| 327/327 [03:17<00:00,  2.09it/s]

Subset Accuracy (exact match): 0.3926940639269406
Micro Precision: 0.7445255474452555
Micro Recall: 0.408
Micro F1: 0.5271317829457365
Macro Precision: 0.3645331232233346
Macro Recall: 0.190038946049875
Macro F1: 0.21629221668560772

Per-class report:
               precision    recall  f1-score   support

         Joy       0.76      0.59      0.67        74
       Trust       1.00      0.03      0.06        30
        Fear       0.00      0.00      0.00         5
    Surprise       0.00      0.00      0.00        11
     Sadness       0.71      0.72      0.72        61
     Disgust       0.00      0.00      0.00        10
       Anger       0.00      0.00      0.00         5
Anticipation       0.81      0.36      0.50        36
     Neutral       0.00      0.00      0.00        18

   micro avg       0.74      0.41      0.53       250
   macro avg       0.36      0.19      0.22       250
weighted avg       0.63      0.41      0.45       250
 samples avg       0.47      0.43      0.44

# First training results:

- Subset Accuracy (exact match): 0.3607305936073059
- Micro Precision: 0.7622950819672131
- Micro Recall: 0.372
- Micro F1: 0.5
- Macro Precision: 0.27744046414356066
- Macro Recall: 0.16385374445483736
- Macro F1: 0.18947683780848593

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.74      | 0.61   | 0.67     | 74      |
| Trust         | 0.00      | 0.00   | 0.00     | 30      |
| Fear          | 0.00      | 0.00   | 0.00     | 5       |
| Surprise      | 0.00      | 0.00   | 0.00     | 11      |
| Sadness       | 0.76      | 0.67   | 0.71     | 61      |
| Disgust       | 0.00      | 0.00   | 0.00     | 10      |
| Anger         | 0.00      | 0.00   | 0.00     | 5       |
| Anticipation  | 1.00      | 0.19   | 0.33     | 36      |
| Neutral       | 0.00      | 0.00   | 0.00     | 18      |

|          | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| **micro avg** | 0.76      | 0.37   | 0.50     | 250     |
| **macro avg** | 0.28      | 0.16   | 0.19     | 250     |
| **weighted avg** | 0.55   | 0.37   | 0.42     | 250     |
| **samples avg**  | 0.42   | 0.39   | 0.40     | 250     |

# Second training results:
- Subset Accuracy (exact match): 0.4063926940639269
- Micro Precision: 0.8015267175572519
- Micro Recall: 0.42
- Micro F1: 0.5511811023622047
- Macro Precision: 0.27762984378170014
- Macro Recall: 0.1942311163622639
- Macro F1: 0.2237746200594498

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.81      | 0.68   | 0.74     | 74      |
| Trust         | 0.00      | 0.00   | 0.00     | 30      |
| Fear          | 0.00      | 0.00   | 0.00     | 5       |
| Surprise      | 0.00      | 0.00   | 0.00     | 11      |
| Sadness       | 0.75      | 0.66   | 0.70     | 61      |
| Disgust       | 0.00      | 0.00   | 0.00     | 10      |
| Anger         | 0.00      | 0.00   | 0.00     | 5       |
| Anticipation  | 0.94      | 0.42   | 0.58     | 36      |
| Neutral       | 0.00      | 0.00   | 0.00     | 18      |

|                  | Precision | Recall | F1-score | Support |
|------------------|-----------|--------|----------|---------|
| **micro avg**    | 0.80      | 0.42   | 0.55     | 250     |
| **macro avg**    | 0.28      | 0.19   | 0.22     | 250     |
| **weighted avg** | 0.56   | 0.42   | 0.47     | 250     |
| **samples avg**  | 0.47   | 0.44   | 0.45     | 250     |



Cross validations - 10 rondes
En principi pandas ho fa automàticament

Una ronda per cada split

----

Mitigar el desbalanceig de classes

